# Homework 2 (Advanced): train the glossary into a small model with LoRA, and compare it with glossary search

Course: AI Engineering Practice Course 2026 (Matsuo-Iwasawa Lab). Repo: `machew-b/AI-Engineering-2026-Autumn`, folder `Homework-02`.

**What this notebook does**

1. Builds Q&A data from the course glossary (`data/glossary.json`) and a test set of 58 questions: the wording seen in training, a new wording, an alias, glossary terms left out of training, and terms that are not in the glossary.
2. Asks the base model `Qwen/Qwen3.5-0.8B` the 58 questions (before training).
3. Trains a LoRA adapter on the glossary Q&A, then asks the same 58 questions again (after training).
4. Puts the base model, the trained model and a plain glossary search side by side.

**How to run**: Runtime > Change runtime type > T4 GPU, then run the cells from the top. It takes about 20 minutes on a T4 (training about 10). Training settings are the course notebook's, except the number of epochs.

**Two runs**: run A (default) trains on the glossary only. Run B (optional) sets `RUN_NAME = "run_b"` and `INCLUDE_ABSTAIN = True`, which adds examples whose answer is "not in the glossary". Use Runtime > Restart session before run B.

**At the end**: a zip with `result.json`, `eval_before.json` and `eval_after.json` is downloaded. Put the unzipped folder under `Homework-02/results/` (see GUIDE.md). Then delete the runtime.

## 0. Check the GPU and install the pinned libraries

Same versions as the course notebook (they were tested together). The red pip `ERROR: pip's dependency resolver ...` lines about `cudf` and `pyarrow` can be ignored.

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU runtimes have no nvidia-smi
    gpu_info = ""
print(gpu_info or "No GPU found. Runtime > Change runtime type > T4 GPU, then run this cell again.")

In [ ]:
# Versions tested on a Colab T4 (the course notebook uses the same set). torch is the one that comes with Colab.
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 needs transformers 5
    "peft": "0.21.1",           # LoRA
    "trl": "1.14.2",            # SFTTrainer
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab's torchao is incompatible with peft 0.21 and breaks training; it is not needed
%pip install -q {spec}

import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. Settings

`RUN_NAME` and `INCLUDE_ABSTAIN` choose run A or run B. Everything else is recorded in `result.json`. Generation is greedy (`do_sample: False`) so that before and after are measured under the same conditions.

`num_train_epochs` is 8 (the course exercise uses 20 for 14 short examples). Here there are 156 examples for 52 terms (3 wordings per term), so 8 passes means each definition is seen about 24 times.

In [ ]:
from pathlib import Path
import json, time

RUN_NAME = "run_a"          # run B: "run_b"
INCLUDE_ABSTAIN = False     # run B: True (adds 48 "not in the glossary" examples: 24 unknown terms x 2 wordings)
REPO_RAW = "https://raw.githubusercontent.com/machew-b/AI-Engineering-2026-Autumn/main/Homework-02"

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",   # fixed so every run uses the same weights
    "seed": 42,
    "num_train_epochs": 8,
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 128,   # the gold phrases sit in the first sentence of each definition, so 128 tokens is enough
    "gen": {"do_sample": False},
    "work_dir": "/content/glossary_lab",
}
WORK = Path(CONFIG["work_dir"]) / RUN_NAME
WORK.mkdir(parents=True, exist_ok=True)
print(RUN_NAME, "| include_abstain =", INCLUDE_ABSTAIN)

## 2. Get the data code from the repo and build the data

The notebook downloads `app/glossary_data.py` (data builder), `app/scoring.py` (scoring), `app/retrieval.py` (glossary search) and `data/glossary.json` from the repo, so the offline tests and this notebook use the same code. If the download fails, the cell asks you to upload the files from your `Homework-02` folder.

**Look at**: the counts (`train=156` for run A, `204` for run B; `test=58`) and the sample training example.

In [ ]:
import sys, urllib.request
HW = Path("/content/hw02")
FILES = ["app/__init__.py", "app/glossary_data.py", "app/scoring.py", "app/retrieval.py", "data/glossary.json"]
for rel in FILES:
    (HW / rel).parent.mkdir(parents=True, exist_ok=True)
missing = []
for rel in FILES:
    try:
        urllib.request.urlretrieve(f"{REPO_RAW}/{rel}", HW / rel)
    except Exception as e:
        print("download failed:", rel, type(e).__name__, e)
        missing.append(rel)
if missing:
    from google.colab import files
    print("Upload these files from your Homework-02 folder:", [Path(m).name for m in missing])
    by_name = {Path(m).name: m for m in missing}
    for name, content in files.upload().items():
        (HW / by_name[name]).write_bytes(content)

sys.path.insert(0, str(HW))
for m in [m for m in sys.modules if m == "app" or m.startswith("app.")]:
    del sys.modules[m]
from app import glossary_data as gd, scoring, retrieval

data = gd.build_dataset(include_abstain=INCLUDE_ABSTAIN)
train, valid, test, gold = data["train"], data["valid"], data["test"], data["gold"]
print(json.dumps(data["meta"], ensure_ascii=False))
print("sample training example:", json.dumps(train[0]["messages"][1:], ensure_ascii=False))

## 3. Load the base model and ask the 58 questions (before training)

The prompt (`gd.build_messages`) is the same one used for training and for the after-training evaluation, so the only difference between before and after is the adapter. The system prompt tells the model to answer with the glossary definition and to say "not in the glossary" for other terms, but the glossary itself is **not** in the prompt: the model has to answer from what it knows.

**Look at**: the summary line (`seen_rate`, `reworded_rate`, ... and `unknown_rate`, the share of unknown terms where it said "not in the glossary") and the answers in the table. Evaluation takes about 3 to 4 minutes. Warnings about `HF Hub` or a "reference PyTorch implementation" are harmless.

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
pd.set_option("display.max_colwidth", 90)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| weight memory (GB):", round(model.get_memory_footprint() / 1e9, 2))

def generate(model_, question):
    prompt = tok.apply_chat_template(gd.build_messages(question), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model_.device)
    with torch.no_grad():
        out = model_.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def run_eval(model_, label):
    summary, rows = scoring.evaluate(lambda q: generate(model_, q), test, gold, label)
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=1), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

def show(rows):
    display(pd.DataFrame([{"id": r["id"], "answer": r["answer"], "correct": "yes" if r["ok"] else "no",
                           "said not in glossary": "yes" if r["abstained"] else "no"} for r in rows]))

before_summary, before_rows = run_eval(model, "before")
show(before_rows)

## 4. Train with LoRA

Same LoRA settings as the course notebook (r 16, alpha 32, lr 2e-4, cosine schedule, batch 4 x accumulation 2). The data is the glossary Q&A in `messages` format, so it goes straight into `Dataset.from_list`. `valid` (another wording, never trained on) is passed along like in the course notebook but is not used to pick anything.

**Look at**: `trainable params ... trainable%` (a few percent), and the Training Loss table (every 10 steps). It should fall steadily. If it does not move, check that `CONFIG` was not changed by accident. If the session stops midway, run sections 0 to 3 again and change `trainer.train()` to `trainer.train(resume_from_checkpoint=True)` (checkpoints are saved every 10 steps).

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds, valid_ds = Dataset.from_list(train), Dataset.from_list(valid)
model.config.use_cache = False
# Qwen3.5: 18 of 24 layers are linear attention (no q/k/v/o_proj); the MLP projections exist in every layer
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4,
                gradient_accumulation_steps=2, learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"],
                logging_steps=10, save_steps=10, save_total_limit=2, fp16=False, max_length=CONFIG["max_seq_len"], packing=False,
                report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
n_trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in trainer.model.parameters())
TRAINABLE = f"{n_trainable:,} of {n_all:,} ({100 * n_trainable / n_all:.2f}%)"

t0 = time.time()
trainer.train()
train_seconds = time.time() - t0
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
print(f"training time: {train_seconds:.0f} s | steps: {trainer.state.global_step} | epochs: {trainer.state.epoch:.1f} | last logged loss: {losses[-1] if losses else None}")
trainer.model.save_pretrained(WORK / "adapter")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()   # dropout off, so the evaluation output does not wobble

## 5. Ask the same 58 questions after training, and compare with glossary search

Same questions, same prompt, same greedy generation. The glossary search is the plain lookup in `app/retrieval.py` (longest term or alias found in the question, no tuning).

**Look at**

- The comparison table: which question types improved, and whether "Not in glossary" went up.
- `regressed`: questions the base model answered correctly but the trained model does not.
- The two rows to read yourself: `heldout` (terms deliberately left out of training) and `unknown` (terms that are not in the glossary). Does the trained model say "not in the glossary", or invent a definition?

In [ ]:
after_summary, after_rows = run_eval(tuned, "after")
ret_summary, ret_rows = retrieval.evaluate_retrieval(test, gold, gd.load_terms())

def pct(x): return "n/a" if x is None else f"{round(x * 100)}%"
labels = {"seen": "Seen wording", "reworded": "New wording", "alias": "Alias", "heldout": "Left out of training",
          "unknown": "Not in glossary (abstain = correct)", "overall": "All questions"}
display(pd.DataFrame([{"Question type": lab, "Base model": pct(before_summary[f"{k}_rate"]), f"{RUN_NAME} (trained)": pct(after_summary[f"{k}_rate"]),
                       "Glossary search": pct(ret_summary[f"{k}_rate"])} for k, lab in labels.items()]))
print("taught terms answered with 'not in the glossary': base", pct(before_summary["over_abstain_rate"]), "| trained", pct(after_summary["over_abstain_rate"]))
regressed = [b["id"] for b, a in zip(before_rows, after_rows) if b["ok"] and not a["ok"]]
print("regressed:", regressed)
show([r for r in after_rows if r["category"] in ("heldout", "unknown")])

## 6. Save the results

Writes `result.json` next to `eval_before.json` and `eval_after.json`, zips the three files and starts the download. Put the unzipped folder (`run_a` or `run_b`) under `Homework-02/results/`. If the download does not start, right-click the zip in the file list on the left and choose Download.

After that: Runtime > Disconnect and delete runtime (the free GPU time is limited).

In [ ]:
result = {
    "run": RUN_NAME, "include_abstain": INCLUDE_ABSTAIN,
    "method": "LoRA (no quantization) on glossary Q&A",
    "config": {k: v for k, v in CONFIG.items() if k != "work_dir"},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "data": data["meta"],
    "training": {"seconds": round(train_seconds, 1), "steps": trainer.state.global_step, "epochs": trainer.state.epoch,
                 "final_loss": losses[-1] if losses else None, "trainable": TRAINABLE},
    "before": before_summary, "after": after_summary, "retrieval": ret_summary,
    "regressed": regressed,
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=1), encoding="utf-8")

import zipfile
zip_path = f"/content/{RUN_NAME}_results.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for name in ["result.json", "eval_before.json", "eval_after.json"]:
        z.write(WORK / name, f"{RUN_NAME}/{name}")
print("saved:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Download did not start:", type(e).__name__, "- use the file list on the left.")